# Page Replacement Algorithm Simulator
**Operating Systems Lab Assignment — Virtual Memory / Page Replacement**

**Algorithms:** FIFO, LRU, Optimal  
**Purpose:** Simulate page replacement, display step-by-step frame states, calculate page faults/hits and ratios, and compare results.

> Enter your own name and roll number before submission. Run every code cell so the notebook contains visible outputs.

## 1. Algorithm overview

- **FIFO (First-In, First-Out):** Replaces the page that entered memory earliest.
- **LRU (Least Recently Used):** Replaces the page that has not been used for the longest time in the past.
- **Optimal:** Replaces the page whose next use is farthest in the future, or which will not be used again. This is a benchmark because a real OS cannot know future references exactly.

A **page hit** occurs when the referenced page is already in a frame. A **page fault** occurs when it is not present.

## 2. Input validation and helper functions

In [11]:
from typing import List, Dict, Any
import pandas as pd

def parse_reference_string(raw: str) -> List[int]:
    """Parse comma- or whitespace-separated non-negative integer pages."""
    parts = raw.replace(",", " ").split()
    if not parts:
        raise ValueError("Please enter at least one page number.")
    try:
        pages = [int(part) for part in parts]
    except ValueError:
        raise ValueError("Invalid reference string: use numeric page values separated by commas or spaces.")
    if any(page < 0 for page in pages):
        raise ValueError("Page numbers must be non-negative integers.")
    return pages

def validate_num_frames(raw: str) -> int:
    try:
        frames = int(raw)
    except ValueError:
        raise ValueError("Number of frames must be a whole number.")
    if frames < 1:
        raise ValueError("Number of frames must be at least 1.")
    return frames

def make_trace_row(step, page, frames, result, replaced):
    return {
        "Step": step,
        "Page": page,
        "Frames": frames.copy(),
        "Result": result,
        "Page Replaced": "-" if replaced is None else replaced
    }

## 3. FIFO algorithm

In [12]:
def fifo(reference: List[int], num_frames: int):
    """Return (page_faults, page_hits, trace) for FIFO."""
    frames = []
    next_to_replace = 0
    faults = hits = 0
    trace = []

    for step, page in enumerate(reference, start=1):
        replaced = None
        if page in frames:
            hits += 1
            result = "Hit"
        else:
            faults += 1
            result = "Fault"
            if len(frames) < num_frames:
                frames.append(page)
            else:
                replaced = frames[next_to_replace]
                frames[next_to_replace] = page
                next_to_replace = (next_to_replace + 1) % num_frames
        trace.append(make_trace_row(step, page, frames, result, replaced))

    return faults, hits, trace

## 4. LRU algorithm

In [13]:
def lru(reference: List[int], num_frames: int):
    """Return (page_faults, page_hits, trace) for LRU."""
    frames = []
    last_used = {}
    faults = hits = 0
    trace = []

    for step, page in enumerate(reference, start=1):
        replaced = None
        if page in frames:
            hits += 1
            result = "Hit"
        else:
            faults += 1
            result = "Fault"
            if len(frames) < num_frames:
                frames.append(page)
            else:
                least_recent_page = min(frames, key=lambda p: last_used[p])
                replaced = least_recent_page
                frames[frames.index(least_recent_page)] = page
        last_used[page] = step
        trace.append(make_trace_row(step, page, frames, result, replaced))

    return faults, hits, trace

## 5. Optimal algorithm

In [14]:
def optimal(reference: List[int], num_frames: int):
    """Return (page_faults, page_hits, trace) for Optimal page replacement."""
    frames = []
    faults = hits = 0
    trace = []

    for index, page in enumerate(reference):
        replaced = None
        if page in frames:
            hits += 1
            result = "Hit"
        else:
            faults += 1
            result = "Fault"
            if len(frames) < num_frames:
                frames.append(page)
            else:
                future = reference[index + 1:]
                # If multiple pages are never used again, choose the one
                # appearing first in the current frame list (assignment rule).
                next_use = {}
                for frame_page in frames:
                    try:
                        next_use[frame_page] = future.index(frame_page)
                    except ValueError:
                        next_use[frame_page] = float("inf")
                victim = max(frames, key=lambda p: next_use[p])
                replaced = victim
                frames[frames.index(victim)] = page
        trace.append(make_trace_row(index + 1, page, frames, result, replaced))

    return faults, hits, trace

## 6. Display trace and performance summary

In [15]:
def show_trace(name: str, faults: int, hits: int, trace: List[Dict[str, Any]]):
    total = faults + hits
    display_rows = []
    for row in trace:
        display_rows.append({
            "Step": row["Step"],
            "Page Referenced": row["Page"],
            "Frame Contents": " | ".join(map(str, row["Frames"])),
            "Hit/Fault": row["Result"],
            "Page Replaced": row["Page Replaced"]
        })
    print(f"\\n{name} — Step-by-step trace")
    display(pd.DataFrame(display_rows).set_index("Step"))
    print(f"Page faults : {faults}")
    print(f"Page hits   : {hits}")
    print(f"Hit ratio   : {hits / total:.2%}")
    print(f"Fault ratio : {faults / total:.2%}")

def compare_algorithms(reference: List[int], num_frames: int):
    algorithms = {
        "FIFO": fifo,
        "LRU": lru,
        "Optimal": optimal
    }
    results = {}
    rows = []
    for name, algorithm in algorithms.items():
        faults, hits, trace = algorithm(reference, num_frames)
        results[name] = (faults, hits, trace)
        show_trace(name, faults, hits, trace)
        total = len(reference)
        rows.append({
            "Algorithm": name,
            "Frames": num_frames,
            "Page Faults": faults,
            "Page Hits": hits,
            "Hit Ratio": f"{hits / total:.2%}",
            "Fault Ratio": f"{faults / total:.2%}"
        })

    print("\\nFinal comparison")
    display(pd.DataFrame(rows).set_index("Algorithm"))
    return results

## 7. Enter your input and run the simulator

In [16]:
try:
    raw_reference = input("Enter page reference string (comma or space separated): ")
    raw_frames = input("Enter number of frames: ")

    reference_string = parse_reference_string(raw_reference)
    number_of_frames = validate_num_frames(raw_frames)

    print("\\nReference string:", reference_string)
    print("Number of frames:", number_of_frames)
    results = compare_algorithms(reference_string, number_of_frames)
except ValueError as error:
    print("Input error:", error)

Enter page reference string (comma or space separated): 1 2 3 1 4 2 5
Enter number of frames: 3
\nReference string: [1, 2, 3, 1, 4, 2, 5]
Number of frames: 3
\nFIFO — Step-by-step trace


,Page Referenced,Frame Contents,Hit/Fault,Page Replaced
Step,,,,
1,1,1,Fault,-
2,2,1 | 2,Fault,-
3,3,1 | 2 | 3,Fault,-
4,1,1 | 2 | 3,Hit,-
5,4,4 | 2 | 3,Fault,1
6,2,4 | 2 | 3,Hit,-
7,5,4 | 5 | 3,Fault,2


Page faults : 5
Page hits   : 2
Hit ratio   : 28.57%
Fault ratio : 71.43%
\nLRU — Step-by-step trace


,Page Referenced,Frame Contents,Hit/Fault,Page Replaced
Step,,,,
1,1,1,Fault,-
2,2,1 | 2,Fault,-
3,3,1 | 2 | 3,Fault,-
4,1,1 | 2 | 3,Hit,-
5,4,1 | 4 | 3,Fault,2
6,2,1 | 4 | 2,Fault,3
7,5,5 | 4 | 2,Fault,1


Page faults : 6
Page hits   : 1
Hit ratio   : 14.29%
Fault ratio : 85.71%
\nOptimal — Step-by-step trace


,Page Referenced,Frame Contents,Hit/Fault,Page Replaced
Step,,,,
1,1,1,Fault,-
2,2,1 | 2,Fault,-
3,3,1 | 2 | 3,Fault,-
4,1,1 | 2 | 3,Hit,-
5,4,4 | 2 | 3,Fault,1
6,2,4 | 2 | 3,Hit,-
7,5,5 | 2 | 3,Fault,4


Page faults : 5
Page hits   : 2
Hit ratio   : 28.57%
Fault ratio : 71.43%
\nFinal comparison


,Frames,Page Faults,Page Hits,Hit Ratio,Fault Ratio
Algorithm,,,,,
FIFO,3,5,2,28.57%,71.43%
LRU,3,6,1,14.29%,85.71%
Optimal,3,5,2,28.57%,71.43%


## 8. Part B — Manual tracing

Use the assignment reference string **7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2** with **3 frames**. Run this cell to generate the exact tables to copy into the manual-tracing section.

In [17]:
part_b_reference = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2]
part_b_frames = 3
part_b_results = compare_algorithms(part_b_reference, part_b_frames)

\nFIFO — Step-by-step trace


,Page Referenced,Frame Contents,Hit/Fault,Page Replaced
Step,,,,
1,7,7,Fault,-
2,0,7 | 0,Fault,-
3,1,7 | 0 | 1,Fault,-
4,2,2 | 0 | 1,Fault,7
5,0,2 | 0 | 1,Hit,-
6,3,2 | 3 | 1,Fault,0
7,0,2 | 3 | 0,Fault,1
8,4,4 | 3 | 0,Fault,2
9,2,4 | 2 | 0,Fault,3


Page faults : 10
Page hits   : 3
Hit ratio   : 23.08%
Fault ratio : 76.92%
\nLRU — Step-by-step trace


,Page Referenced,Frame Contents,Hit/Fault,Page Replaced
Step,,,,
1,7,7,Fault,-
2,0,7 | 0,Fault,-
3,1,7 | 0 | 1,Fault,-
4,2,2 | 0 | 1,Fault,7
5,0,2 | 0 | 1,Hit,-
6,3,2 | 0 | 3,Fault,1
7,0,2 | 0 | 3,Hit,-
8,4,4 | 0 | 3,Fault,2
9,2,4 | 0 | 2,Fault,3


Page faults : 9
Page hits   : 4
Hit ratio   : 30.77%
Fault ratio : 69.23%
\nOptimal — Step-by-step trace


,Page Referenced,Frame Contents,Hit/Fault,Page Replaced
Step,,,,
1,7,7,Fault,-
2,0,7 | 0,Fault,-
3,1,7 | 0 | 1,Fault,-
4,2,2 | 0 | 1,Fault,7
5,0,2 | 0 | 1,Hit,-
6,3,2 | 0 | 3,Fault,1
7,0,2 | 0 | 3,Hit,-
8,4,2 | 4 | 3,Fault,0
9,2,2 | 4 | 3,Hit,-


Page faults : 7
Page hits   : 6
Hit ratio   : 46.15%
Fault ratio : 53.85%
\nFinal comparison


,Frames,Page Faults,Page Hits,Hit Ratio,Fault Ratio
Algorithm,,,,,
FIFO,3,10,3,23.08%,76.92%
LRU,3,9,4,30.77%,69.23%
Optimal,3,7,6,46.15%,53.85%


## 9. Part C — Experiment 1: Belady's anomaly

Reference string: **1 2 3 4 1 2 5 1 2 3 4 5**. Compare 3 frames with 4 frames.

In [18]:
belady_reference = [1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5]

belady_rows = []
for frame_count in (3, 4):
    for name, algorithm in (("FIFO", fifo), ("LRU", lru), ("Optimal", optimal)):
        faults, hits, _ = algorithm(belady_reference, frame_count)
        belady_rows.append({
            "Algorithm": name,
            "Frames": frame_count,
            "Page Faults": faults,
            "Page Hits": hits
        })

belady_table = pd.DataFrame(belady_rows)
display(belady_table.pivot(index="Algorithm", columns="Frames", values="Page Faults")
        .rename(columns={3: "Faults (3 frames)", 4: "Faults (4 frames)"}))

fifo_3 = belady_table.query("Algorithm == 'FIFO' and Frames == 3")["Page Faults"].iloc[0]
fifo_4 = belady_table.query("Algorithm == 'FIFO' and Frames == 4")["Page Faults"].iloc[0]
print("Belady's anomaly observed for FIFO:", fifo_4 > fifo_3)

Frames,Faults (3 frames),Faults (4 frames)
Algorithm,,
FIFO,9,10
LRU,10,8
Optimal,7,6


Belady's anomaly observed for FIFO: True


### Part C — Experiment 1 answers

1. **Does any algorithm produce more faults with more frames?** FIFO exhibits Belady's anomaly for the supplied reference string if its page-fault count increases when moving from 3 to 4 frames. Check the table produced above.
2. **Why do LRU and Optimal avoid this behaviour?** LRU and Optimal have the stack property: the set of pages held with fewer frames is a subset of the set held with more frames, so increasing frames cannot increase their page-fault count.

## 10. Part C — Experiment 2: Your own inputs

The following examples use 3 frames and 15 references each. Run the cell to calculate the actual results. If your teacher expects you to design your own strings, change the strings and record the resulting counts.

In [20]:
experiment_cases = {
    "LRU better": [1, 2, 3, 1, 2, 4, 1, 2, 3, 1, 2, 4, 1, 2, 3],
    "FIFO better/equal": [1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3]
}
experiment_frames = 3
experiment_rows = []

for case_name, pages in experiment_cases.items():
    fifo_faults, _, _ = fifo(pages, experiment_frames)
    lru_faults, _, _ = lru(pages, experiment_frames)
    experiment_rows.append({
        "Case": case_name,
        "Reference string": " ".join(map(str, pages)),
        "Frames": experiment_frames,
        "FIFO faults": fifo_faults,
        "LRU faults": lru_faults
    })

display(pd.DataFrame(experiment_rows))

,Case,Reference string,Frames,FIFO faults,LRU faults
0,LRU better,1 2 3 1 2 4 1 2 3 1 2 4 1 2 3,3,11,7
1,FIFO better/equal,1 2 3 4 1 2 3 4 1 2 3 4 1 2 3,3,15,15


## 11. Part C — Short answers

1. **Why can Optimal not be implemented in a real operating system?** It requires knowing future page references in advance, which a real operating system cannot know exactly.
2. **What extra hardware or bookkeeping does LRU need compared with FIFO?** LRU must track page usage recency (for example, timestamps/counters or a stack/list) and update that information on each reference. FIFO mainly tracks arrival order.
3. **If the number of frames equals the number of distinct pages in the string, how many faults will every algorithm produce? Why?** Each distinct page causes one fault on its first reference, and no page needs to be evicted. Therefore, the fault count equals the number of distinct pages, and later references are hits.

## 12. Project conclusion

This simulator implements FIFO, LRU and Optimal page replacement and compares them using page faults, page hits, hit ratio and fault ratio. Optimal generally provides a lower-bound benchmark for page faults because it uses knowledge of future references. FIFO is simple but can exhibit Belady's anomaly; LRU uses recent past behaviour to select a victim.

